In [1]:
# !/usr/bin/env python3
import os
import torch
from equiv_dens.training.parse_command_line_arguments import parse_command_line_arguments
from equiv_dens.data.density_dataset import AtomsDensityData
from equiv_dens.utils.grids import cubical_grid, cubical_sampling,\
    spherical_grid, spherical_radial_sampling
import equiv_dens.utils.base as utils
from equiv_dens.training.model_loader import load_model

import numpy as np
from functools import partial
import argparse
%load_ext autoreload
%autoreload 2

/home/mihail/anaconda3/envs/equiv_dens/lib/python3.7/site-packages/pyscf/lib/misc.py:47: H5pyDeprecationWarning: Using default_file_mode other than 'r' is deprecated. Pass the mode to h5py.File() instead.
  h5py.get_config().default_file_mode = 'a'


Use "numpy" for Fourier Transform


In [2]:
class LoadFromFile (argparse.Action):
    def __call__ (self, parser, namespace, values, option_string = None):
        with values as f:
            # parse arguments in the file and store them in the target namespace
            parser.parse_args(f.read().split(), namespace)

In [3]:
args, hyperparam_args = parse_command_line_arguments(arg_file='ethanol_dens_dft8_mae_test.txt')
print('type dtype', type(args.dtype))
args.fix_arguments = True
print('args np dir', args.np_dataset)
# no restart directory specified
directory = args.restart  # load directory name
# load latest checkpoint
checkpoint_path = os.path.join(directory, 'checkpoints')  # checkpoint directory
checkpoint = torch.load(os.path.join(
    checkpoint_path, 'latest_checkpoint.pth'), map_location='cpu')
latest_checkpoint = checkpoint['step']
model_code = checkpoint['ID']  # load ID
step = checkpoint['step']
for arg in vars(checkpoint['args']):
    if args.fix_arguments:
        if arg in hyperparam_args:
            print('loading hyperparam arg', arg)
            setattr(args, arg, getattr(checkpoint['args'], arg))
    else:
        print('loading all arg', arg)
        setattr(args, arg, getattr(checkpoint['args'], arg))
restore = True

args.best_model_path = 'best_' + model_code + '.pth'
print('best_model_path', args.best_model_path)

print('model code:', model_code)
# determine whether GPU is used for training
print('args use gpu', args.use_gpu)
args.use_gpu = args.use_gpu and torch.cuda.is_available()

# load dataset(s)
print("loading density from" + str(args.dens_dataset) + "...")
print("loading atoms from" + args.np_dataset + "...")
args.use_gpu = False
if args.cube_grid:
    grid_origin = args.cube_origin
    grid_extent = np.array([args.cube_extent] * 3)
    grid_fn = partial(cubical_grid, nx=args.cube_size, ny=args.cube_size, nz=args.cube_size,
                      extent=grid_extent,
                      origin=np.array([grid_origin] * 3))
    sampling_fn = cubical_sampling
else:
    grid_fn = partial(spherical_grid, level=2)
    sampling_fn = partial(spherical_radial_sampling, rotate=False)
    grid_origin = 0
    grid_extent = None

dataset = AtomsDensityData(np_path=args.np_dataset, density_path=args.dens_dataset,
                           orbitals_path=args.orbitals_file,
                           density_n_samp=10000000000,
                           required_properties=['density'],
                           center_positions=False,
                           radial_coeffs_file=args.radial_coeffs_file,
                           dtype=args.dtype,
                           grid_fn=grid_fn,
                           sampling_fn=sampling_fn,
                           grid_extent=grid_extent,
                           grid_origin=grid_origin,
                           verbose=args.verbose)

old_model = load_model(args, dataset)

type dtype <class 'torch.dtype'>
args np dir datasets/ethanol_dft_train.npy
loading hyperparam arg activation
loading hyperparam arg order
loading hyperparam arg mixing_order
loading hyperparam arg order_en
loading hyperparam arg mixing_order_en
loading hyperparam arg num_features
loading hyperparam arg num_basis_functions
loading hyperparam arg num_radial_components
loading hyperparam arg num_energy_features
loading hyperparam arg num_modules
loading hyperparam arg num_residual_pre_x
loading hyperparam arg num_residual_post_x
loading hyperparam arg num_residual_pre_vi
loading hyperparam arg num_residual_pre_vj
loading hyperparam arg num_residual_post_v
loading hyperparam arg num_residual_output
loading hyperparam arg num_energy_output
loading hyperparam arg basis_functions
loading hyperparam arg cutoff
loading hyperparam arg orthonormal_basis
loading hyperparam arg expansion_constraint
loading hyperparam arg integral_constraint
loading hyperparam arg integral_scale
loading hyperparam 

In [4]:
ids = np.random.randint(len(dataset), size=(10,))

samples = dataset.get_properties(ids)

properties positions type torch.FloatTensor


In [5]:
samples.keys()

dict_keys(['coords', 'coord_weights', 'density', 'atom_numbers', 'idx', 'positions', 'shifted_positions', '_idx'])

In [6]:
print('true density integral', torch.sum(samples['density'] * samples['coord_weights'], -1))

true density integral tensor([26.0000, 26.0000, 26.0000, 26.0000, 26.0000, 26.0000, 26.0000, 26.0000,
        26.0000, 26.0000])


In [8]:
pred = old_model(samples)
print('pred density integral', torch.sum(pred['density'] * pred['coord_weights'], -1))

Memory allocated 0.0
Memory cached 0.0


/home/mihail/anaconda3/envs/equiv_dens/lib/python3.7/site-packages/torch/cuda/memory.py:346: FutureWarning: torch.cuda.memory_cached has been renamed to torch.cuda.memory_reserved
  FutureWarning)


Memory allocated 0.0
Memory cached 0.0
L0_coeffs comb sum before tensor([25.9802, 25.9763, 25.9743, 25.9750, 25.9712, 25.9694, 25.9721, 25.9775,
        25.9703, 25.9708], grad_fn=<SumBackward1>)
integral scale tensor([1.])
L0_coeffs comb sum after tensor([25.9802, 25.9763, 25.9743, 25.9750, 25.9712, 25.9694, 25.9721, 25.9775,
        25.9703, 25.9708], grad_fn=<SumBackward1>)
density nan tensor(0)
L0 width tensor([[[[5.8462e+01, 6.9837e+02, 1.7855e+02, 1.2218e+01, 7.5918e+00,
           1.6062e+01, 2.3680e+00, 9.8700e-01, 8.8502e-01, 3.3045e-01,
           1.3092e-01]]],


        [[[5.8425e+01, 6.9851e+02, 1.7937e+02, 1.2111e+01, 7.5772e+00,
           1.6063e+01, 2.3531e+00, 9.8394e-01, 8.8611e-01, 3.3079e-01,
           1.3103e-01]]],


        [[[5.8532e+01, 6.9851e+02, 1.7843e+02, 1.2200e+01, 7.6307e+00,
           1.6063e+01, 2.3586e+00, 9.8529e-01, 8.8537e-01, 3.3026e-01,
           1.3113e-01]]],


        [[[5.8356e+01, 6.9847e+02, 1.7890e+02, 1.2182e+01, 7.5983e+00,
        

rbf nan tensor(0)
sph nan tensor(0)
L0 width tensor([[[[11.4998,  3.1739,  1.3476,  0.5667,  0.0123]]],


        [[[11.5064,  3.1691,  1.3477,  0.5667,  0.0123]]],


        [[[11.5135,  3.1909,  1.3480,  0.5667,  0.0120]]],


        [[[11.4700,  3.2022,  1.3478,  0.5666,  0.0124]]],


        [[[11.4926,  3.2361,  1.3484,  0.5667,  0.0117]]],


        [[[11.5085,  3.1983,  1.3477,  0.5666,  0.0125]]],


        [[[11.4831,  3.2006,  1.3478,  0.5667,  0.0121]]],


        [[[11.5420,  3.1795,  1.3475,  0.5667,  0.0123]]],


        [[[11.4991,  3.1991,  1.3480,  0.5667,  0.0120]]],


        [[[11.5228,  3.1786,  1.3476,  0.5667,  0.0124]]]],
       grad_fn=<MulBackward0>)
L0 width negative tensor(0)
rbf nan tensor(0)
sph nan tensor(0)
density nan tensor(0)
Memory allocated 0.0
Memory cached 0.0
pred density integral tensor([25.9805, 25.9768, 25.9746, 25.9754, 25.9713, 25.9698, 25.9723, 25.9778,
        25.9706, 25.9712], grad_fn=<SumBackward1>)


In [9]:
print('pred keys', pred.keys())

pred keys dict_keys(['coords', 'coord_weights', 'density', 'atom_numbers', 'idx', 'positions', 'shifted_positions', '_idx', 'distances', 'directions', 'sph', 'sph_repr', 'spherical_coeffs', 'radial_width', 'radial_scale', 'L_dict', 'L0_coeffs'])


In [10]:
print('spherical coeffs', pred['spherical_coeffs'])

spherical coeffs [{(6, 0): tensor([[[[ 1.1728,  0.4210,  0.9388, -0.8778,  0.6821,  1.0196,  1.8027,
            0.4562,  1.4075,  0.0594,  0.0679]]],


        [[[ 1.1717,  0.4160,  0.9435, -0.8777,  0.6765,  1.0180,  1.8052,
            0.4562,  1.4185,  0.0787,  0.0796]]],


        [[[ 1.1715,  0.4178,  0.9358, -0.8710,  0.6783,  1.0198,  1.8064,
            0.4565,  1.4122,  0.0624,  0.0716]]],


        [[[ 1.1707,  0.4184,  0.9400, -0.8769,  0.6807,  1.0177,  1.8013,
            0.4563,  1.4115,  0.0680,  0.0742]]],


        [[[ 1.1729,  0.4198,  0.9395, -0.8814,  0.6823,  1.0160,  1.8036,
            0.4579,  1.4100,  0.0711,  0.0773]]],


        [[[ 1.1710,  0.4194,  0.9396, -0.8785,  0.6824,  1.0174,  1.8011,
            0.4568,  1.4096,  0.0687,  0.0758]]],


        [[[ 1.1697,  0.4182,  0.9366, -0.8742,  0.6824,  1.0172,  1.8031,
            0.4569,  1.4107,  0.0672,  0.0764]]],


        [[[ 1.1663,  0.4150,  0.9367, -0.8689,  0.6817,  1.0182,  1.8007,
            0.455

In [11]:
print('L0 mean', torch.mean(pred['L0_coeffs'], 0))
print('L0 variance', torch.std(pred['L0_coeffs'], 0))
print('L0 sum', torch.sum(pred['L0_coeffs'], 1))

L0 mean tensor([ 1.1071e+00,  7.8806e-02,  5.3570e-01, -7.7195e-01,  4.9168e-01,
         6.2422e-01,  2.4528e+00,  8.4951e-02,  1.5373e+00,  1.4442e-02,
         4.8471e-03,  1.1124e+00,  7.8277e-02,  5.3282e-01, -7.8947e-01,
         4.7838e-01,  6.4023e-01,  2.5313e+00,  8.3117e-02,  1.5337e+00,
         1.5898e-02,  5.0546e-03,  5.8763e-01,  1.1705e-01,  1.0811e+00,
         2.4443e+00,  5.4756e-01,  1.7621e+00,  8.6971e-02,  1.0777e-01,
         1.5716e+00, -1.6612e-01, -1.8729e-02,  1.5427e-01,  5.9716e-01,
         1.0749e-01,  6.5712e-02, -1.0191e-04,  1.5434e-01,  5.9645e-01,
         1.0771e-01,  6.5301e-02, -1.0565e-04,  1.4965e-01,  5.9229e-01,
         1.0975e-01,  6.2540e-02, -1.6882e-04,  1.5053e-01,  5.9350e-01,
         1.0869e-01,  6.2457e-02, -1.5126e-04,  1.5003e-01,  5.9298e-01,
         1.0908e-01,  6.3308e-02, -1.5274e-04,  1.3160e-01,  5.2219e-01,
         1.5705e-01,  6.7474e-02,  5.0284e-05], grad_fn=<MeanBackward1>)
L0 variance tensor([1.8289e-03, 5.3076e-04,

In [12]:
L0_scale = []
L0_width = []

for i in range(len(pred['radial_width'])):
    for orb in pred['radial_width'][i].keys():
        if orb[1] == 0:
            L0_scale.append(pred['radial_scale'][i][orb])
            L0_width.append(pred['radial_width'][i][orb])

L0_scale = torch.cat(L0_scale, -1).squeeze()
L0_width = torch.cat(L0_width, -1).squeeze()
print('L0_scale shape', L0_scale.shape)
print('L0_width shape', L0_width.shape)

print('L0_scale mean', torch.mean(L0_scale, 0))
print('L0_width mean', torch.mean(L0_width, 0))
print('L0_scale std', torch.std(L0_scale, 0))
print('L0_width std', torch.std(L0_width, 0))


L0_scale shape torch.Size([10, 63])
L0_width shape torch.Size([10, 63])
L0_scale mean tensor([-0.0547, -0.8116, -0.4298, -0.1196, -0.2775, -0.3868,  0.3603, -0.8138,
         0.0887, -0.7925, -0.9361, -0.0549, -0.8201, -0.4297, -0.1047, -0.2791,
        -0.3793,  0.3785, -0.8194,  0.0663, -0.7879, -0.9421, -0.2477, -0.6784,
        -2.0491,  0.4582, -1.6126,  0.1585, -0.7951, -0.3781,  0.4529, -0.5024,
        -0.9511, -0.6955, -1.6036, -0.6098, -0.7577, -0.9931, -0.6953, -1.6032,
        -0.6094, -0.7588, -0.9927, -0.7014, -1.6036, -0.6082, -0.7551, -0.9881,
        -0.7011, -1.6055, -0.6093, -0.7543, -0.9888, -0.7020, -1.6048, -0.6090,
        -0.7538, -0.9892, -0.7294, -1.5764, -0.5802, -0.7790, -1.0224],
       grad_fn=<MeanBackward1>)
L0_width mean tensor([-0.9476,  0.8921,  0.4687, -0.7467, -0.6264,  0.9859, -0.0592, -0.2082,
         0.8278,  0.7229,  0.7245, -0.9475,  0.8917,  0.4741, -0.7403, -0.6147,
         0.9863, -0.0682, -0.2033,  0.8343,  0.7215,  0.7303, -0.8128,  0.97

In [14]:
print(checkpoint['data_split_indices'])

{'train': array([521, 737, 740, 660, 411, 678, 626, 513, 859, 136, 811,  76, 636,
       973, 938, 899, 280, 883, 761, 319, 549, 174, 371, 527, 210, 235,
       101, 986, 902, 947, 346, 139, 621, 499, 370, 198, 687, 584, 901,
        59, 328,  96, 312, 974, 299, 277, 924, 601, 439, 837, 570, 879,
       261, 578,  23,  30, 617,  10, 221, 820, 296,  54, 542, 209, 604,
       692, 662, 866,  70, 543, 107, 493, 590, 741, 292, 289, 652,  39,
       589, 307, 679,  66, 275,  67, 318, 548, 998, 714, 753, 327, 382,
       451, 522, 218, 787, 436, 764,  88,  63, 826, 716, 351, 936, 256,
       635, 644, 554, 959, 168, 917, 528, 823, 985, 816,  86, 432, 184,
       978, 534, 294, 892, 425, 713, 260, 237, 559, 583, 445, 867, 800,
       599, 849, 265, 995, 529,  55, 120, 215,  25,  72,  44, 247, 721,
       281, 893, 914, 810, 244, 822, 321, 643, 158, 977, 429, 941, 462,
       309, 697,  60, 884, 595, 767, 649, 650, 865, 668, 298, 689, 314,
       310, 361, 479, 110, 989, 486, 363, 254, 259, 80

In [52]:
ids = np.arange(10)
model.eval()
with torch.no_grad():
    samples = dataset.get_properties(ids)
    pred = model(samples)
    L0_coeffs = [{} for i in range(len(pred['spherical_coeffs']))]
    L0_widths = [{} for i in range(len(pred['spherical_coeffs']))]
    L0_scales = [{} for i in range(len(pred['spherical_coeffs']))]
    print('ids', ids)
    for j in range(len(pred['spherical_coeffs'])):
        for orb in pred['spherical_coeffs'][j].keys():
            if orb[1] == 0:
                L0_coeffs[j][orb] = pred['spherical_coeffs'][j][orb].clone()
                L0_scales[j][orb] = pred['radial_scale'][j][orb].clone()
                L0_widths[j][orb] = pred['radial_width'][j][orb].clone()
    for i in range(1, int(len(dataset)/10)):
        ids = np.arange(i*10, (i+1) * 10)
        print('ids', ids)
        samples = dataset.get_properties(ids)
        pred = model(samples)
        for j in range(len(pred['spherical_coeffs'])):
            for orb in pred['spherical_coeffs'][j].keys():
                if orb[1] == 0:
                    #print('coeffs shape before', L0_coeffs[j][orb].shape)
                    L0_coeffs[j][orb] = torch.cat((L0_coeffs[j][orb], pred['spherical_coeffs'][j][orb]), 0)
                    L0_scales[j][orb] = torch.cat((L0_scales[j][orb], pred['radial_scale'][j][orb]), 0)
                    L0_widths[j][orb] = torch.cat((L0_widths[j][orb], pred['radial_width'][j][orb]), 0)
        

properties positions type torch.FloatTensor
Memory allocated 0.0
Memory cached 0.0
Memory allocated 0.0
Memory cached 0.0


KeyboardInterrupt: 

In [21]:
print(L0_coeffs[0][(6,0)].shape)

torch.Size([1000, 1, 1, 11])


In [31]:
for j in range(len(L0_coeffs)):
    print('atom', j)
    for orb in pred['spherical_coeffs'][j].keys():
        if orb[1] == 0:
            print('orb', orb)
            print('coeffs mean', torch.mean(L0_coeffs[j][orb],0).squeeze())
            print('scales mean', torch.mean(L0_scales[j][orb],0).squeeze())
            print('widths mean', torch.mean(L0_widths[j][orb],0).squeeze())
            print('coeffs std', torch.std(L0_coeffs[j][orb],0).squeeze())
            print('scales std', torch.std(L0_scales[j][orb],0).squeeze())
            print('widths std', torch.std(L0_widths[j][orb],0).squeeze())

atom 0
orb (6, 0)
coeffs mean tensor([ 1.1699,  0.4176,  0.9394, -0.8756,  0.6810,  1.0179,  1.8026,  0.4563,
         1.4137,  0.0722,  0.0779])
scales mean tensor([-0.0534, -0.8111, -0.4300, -0.1197, -0.2779, -0.3878,  0.3610, -0.8139,
         0.0887, -0.7939, -0.9359])
widths mean tensor([-0.9476,  0.8921,  0.4691, -0.7468, -0.6260,  0.9859, -0.0602, -0.2089,
         0.8280,  0.7231,  0.7250])
coeffs std tensor([0.0024, 0.0024, 0.0027, 0.0045, 0.0026, 0.0015, 0.0024, 0.0011, 0.0032,
        0.0065, 0.0045])
scales std tensor([0.0024, 0.0016, 0.0023, 0.0015, 0.0009, 0.0028, 0.0019, 0.0020, 0.0019,
        0.0038, 0.0006])
widths std tensor([7.4517e-05, 1.7080e-04, 2.4640e-03, 9.5970e-04, 1.7134e-03, 6.5028e-05,
        2.3086e-03, 1.3233e-03, 6.8028e-04, 9.6794e-04, 1.3906e-03])
atom 1
orb (6, 0)
coeffs mean tensor([ 1.1759,  0.4343,  0.9351, -0.8816,  0.6623,  1.0302,  1.8354,  0.4593,
         1.4413,  0.0808,  0.0919])
scales mean tensor([-0.0531, -0.8201, -0.4304, -0.1060, -0.2

In [30]:
expansion_model = model.property_models['density']

for i in range(len(pred['spherical_coeffs'])):
    sph_coeff = pred['spherical_coeffs'][i][(int(samples['atom_numbers'][0,i]),0)][0]
    radial_scale = pred['radial_scale'][i][(int(samples['atom_numbers'][0,i]),0)][0]
    init_scale = expansion_model.init_scale(i, (int(samples['atom_numbers'][0,i]),0))[0]
    zero_scale = (init_scale != 0).to(radial_scale)
    scale = (radial_scale + init_scale) * zero_scale
    print('coeffs * scale', sph_coeff * scale)
print('L0 coeffs', pred['L0_coeffs'][0])

coeffs * scale tensor([[[ 1.1096,  0.0793,  0.5329, -0.7662,  0.4878,  0.6198,  2.4599,
           0.0846,  1.5378,  0.0152,  0.0051]]])
coeffs * scale tensor([[[ 1.1150,  0.0781,  0.5311, -0.7817,  0.4736,  0.6351,  2.5379,
           0.0798,  1.5431,  0.0188,  0.0060]]])
coeffs * scale tensor([[[ 0.5856,  0.1154,  1.0820,  2.4438,  0.5487,  1.7584,  0.0886,
           0.1117,  1.5725, -0.1695, -0.0185]]])
coeffs * scale tensor([[[ 1.6361e-01,  6.0425e-01,  9.9335e-02,  5.9981e-02, -5.3290e-05]]])
coeffs * scale tensor([[[ 1.3830e-01,  5.9099e-01,  1.2035e-01,  6.8212e-02, -2.1110e-04]]])
coeffs * scale tensor([[[ 1.4716e-01,  5.9396e-01,  1.1255e-01,  6.5747e-02, -8.5729e-05]]])
coeffs * scale tensor([[[ 1.3665e-01,  5.8770e-01,  1.2144e-01,  6.4966e-02, -1.3746e-04]]])
coeffs * scale tensor([[[ 1.3344e-01,  5.8274e-01,  1.2389e-01,  6.6209e-02, -2.2243e-04]]])
coeffs * scale tensor([[[1.3743e-01, 5.1771e-01, 1.5564e-01, 6.6680e-02, 1.3106e-04]]])
L0 coeffs tensor([ 1.1096e+00,  7.93

In [33]:
init_coeffs = {}
init_coeffs['spherical_coeffs'] = L0_coeffs
init_coeffs['radial_scale'] = L0_scales
init_coeffs['radial_width'] = L0_widths

np.save('datasets/augccpvqzjkfit_init_L0_old.npy', init_coeffs, allow_pickle=True)

In [4]:
from equiv_dens.utils.misc import generate_id
from datetime import datetime


# no init coeffs
args, hyperparam_args = parse_command_line_arguments(arg_file='ethanol_dens_dft8_mae.txt')
print('type dtype', type(args.dtype))
args.fix_arguments = True
print('args np dir', args.np_dataset)
# no restart directory specified

model_code = generate_id()
directory = os.path.join(args.save_dir, datetime.utcnow().strftime("%Y-%m-%d_") +
                         model_code)  # generate directory name
# create directories
if not os.path.exists(directory):
    os.makedirs(directory)
# write command line arguments to file (useful for reproducibility)
with open(os.path.join(directory, 'args.txt'), 'w') as f:
    for key in args.__dict__.keys():
        # special case for list input
        if isinstance(args.__dict__[key], list):
            for entry in args.__dict__[key]:
                f.write('--' + key + '=' + str(entry) + "\n")
        else:
            f.write('--' + key + '=' + str(args.__dict__[key]) + "\n")
checkpoint = None
latest_checkpoint = 0
step = 0
restore = False
data_split_indices = None
# restarts run from latest checkpoint


args.best_model_path = 'best_' + model_code + '.pth'
print('best_model_path', args.best_model_path)

print('model code:', model_code)
# determine whether GPU is used for training
print('args use gpu', args.use_gpu)
args.use_gpu = args.use_gpu and torch.cuda.is_available()
print("loading density from" + str(args.dens_dataset) + "...")
print("loading atoms from" + args.np_dataset + "...")
args.use_gpu = False
if args.cube_grid:
    grid_origin = args.cube_origin
    grid_extent = np.array([args.cube_extent] * 3)
    grid_fn = partial(cubical_grid, nx=args.cube_size, ny=args.cube_size, nz=args.cube_size,
                      extent=grid_extent,
                      origin=np.array([grid_origin] * 3))
    sampling_fn = cubical_sampling
else:
    grid_fn = partial(spherical_grid, level=2)
    sampling_fn = partial(spherical_radial_sampling, rotate=False)
    grid_origin = 0
    grid_extent = None

dataset = AtomsDensityData(np_path=args.np_dataset, density_path=args.dens_dataset,
                           orbitals_path=args.orbitals_file,
                           density_n_samp=10000000000,
                           required_properties=['density'],
                           center_positions=False,
                           radial_coeffs_file=args.radial_coeffs_file,
                           # L0_coeffs_file=args.L0_coeffs_file,
                           dtype=args.dtype,
                           grid_fn=grid_fn,
                           sampling_fn=sampling_fn,
                           grid_extent=grid_extent,
                           grid_origin=grid_origin,
                           verbose=args.verbose)

model = load_model(args, dataset)

type dtype <class 'torch.dtype'>
args np dir datasets/ethanol_dft_train.npy
best_model_path best_FkGgU0zb.pth
model code: FkGgU0zb
args use gpu True
loading density fromdatasets/ethanol_pyscf_def2svp_dft_f.npy...
loading atoms fromdatasets/ethanol_dft_train.npy...
Starting atomsdata density init
Some variables
atoms keys dict_keys(['positions', 'energy', 'forces', 'atom_numbers', 'atom_types'])
level 2
dataset init grid_spec type torch.FloatTensor
finished init
cg_matrix shape torch.Size([121, 121, 121])
self order [1, 3, 5]
self order [1, 3, 5]
self mixing_order [1, 3, 5]
self mixing_order [1, 3, 5]
orbital_spec [[(6, 11, 0), (6, 8, 1), (6, 6, 2), (6, 4, 3), (6, 3, 4), (6, 2, 5)], [(6, 11, 0), (6, 8, 1), (6, 6, 2), (6, 4, 3), (6, 3, 4), (6, 2, 5)], [(8, 11, 0), (8, 8, 1), (8, 6, 2), (8, 4, 3), (8, 3, 4), (8, 2, 5)], [(1, 5, 0), (1, 4, 1), (1, 4, 2), (1, 3, 3), (1, 2, 4)], [(1, 5, 0), (1, 4, 1), (1, 4, 2), (1, 3, 3), (1, 2, 4)], [(1, 5, 0), (1, 4, 1), (1, 4, 2), (1, 3, 3), (1, 2, 4)], 

In [14]:
ic = np.load('datasets/augccpvqzjkfit_init_L0.npy', allow_pickle=True).item()

print('spherical coeffs O', ic['spherical_coeffs']['O'])
print('width coeffs O', ic['radial_width']['O'])

spherical coeffs O tensor([[ 0.7820,  0.3655, -1.0298,  1.6738, -0.8940,  1.5229,  0.4247,  0.1693,
          1.0828, -0.3339, -0.3808]])
width coeffs O tensor([[-0.8131,  0.9712, -0.4379, -0.9347, -0.5223, -0.5821, -0.8994, -0.1850,
          0.9155,  0.9483,  0.7319]])


In [15]:
ir = np.load('datasets/augccpvqzjkfit_radial_coeffs_df.npy', allow_pickle=True).item()

print('width coeffs O', ir['O'])

width coeffs O [(array([1517.8667506]), array([1.])), (array([489.67952008]), array([1.])), (array([176.72118665]), array([1.])), (array([63.79223314]), array([1.])), (array([25.36649913]), array([1.])), (array([9.91354912]), array([1.])), (array([4.46453066]), array([1.])), (array([1.80177437]), array([1.])), (array([0.80789711]), array([1.])), (array([0.33864327]), array([1.])), (array([0.14194786]), array([1.])), (array([120.16030921]), array([1.])), (array([34.40962247]), array([1.])), (array([12.58114861]), array([1.])), (array([5.06638242]), array([1.])), (array([2.03469271]), array([1.])), (array([0.86092967]), array([1.])), (array([0.36681357]), array([1.])), (array([0.15628709]), array([1.])), (array([19.04306281]), array([1.])), (array([5.80603811]), array([1.])), (array([2.18918416]), array([1.])), (array([0.87794614]), array([1.])), (array([0.35623647]), array([1.])), (array([0.14454693]), array([1.])), (array([3.96585]), array([1.])), (array([1.49085]), array([1.])), (arra

In [18]:
from equiv_dens.utils.misc import generate_id
from datetime import datetime


# no init coeffs
args, hyperparam_args = parse_command_line_arguments(arg_file='ethanol_dens_dft8_mae_init.txt')
print('type dtype', type(args.dtype))
args.fix_arguments = True
print('args np dir', args.np_dataset)
# no restart directory specified

model_code = generate_id()
directory = os.path.join(args.save_dir, datetime.utcnow().strftime("%Y-%m-%d_") +
                         model_code)  # generate directory name
# create directories
if not os.path.exists(directory):
    os.makedirs(directory)
# write command line arguments to file (useful for reproducibility)
with open(os.path.join(directory, 'args.txt'), 'w') as f:
    for key in args.__dict__.keys():
        # special case for list input
        if isinstance(args.__dict__[key], list):
            for entry in args.__dict__[key]:
                f.write('--' + key + '=' + str(entry) + "\n")
        else:
            f.write('--' + key + '=' + str(args.__dict__[key]) + "\n")
checkpoint = None
latest_checkpoint = 0
step = 0
restore = False
data_split_indices = None
# restarts run from latest checkpoint


args.best_model_path = 'best_' + model_code + '.pth'
print('best_model_path', args.best_model_path)

print('model code:', model_code)
# determine whether GPU is used for training
print('args use gpu', args.use_gpu)
args.use_gpu = args.use_gpu and torch.cuda.is_available()
print("loading density from" + str(args.dens_dataset) + "...")
print("loading atoms from" + args.np_dataset + "...")
args.use_gpu = False
if args.cube_grid:
    grid_origin = args.cube_origin
    grid_extent = np.array([args.cube_extent] * 3)
    grid_fn = partial(cubical_grid, nx=args.cube_size, ny=args.cube_size, nz=args.cube_size,
                      extent=grid_extent,
                      origin=np.array([grid_origin] * 3))
    sampling_fn = cubical_sampling
else:
    grid_fn = partial(spherical_grid, level=2)
    sampling_fn = partial(spherical_radial_sampling, rotate=False)
    grid_origin = 0
    grid_extent = None

dataset = AtomsDensityData(np_path=args.np_dataset, density_path=args.dens_dataset,
                           orbitals_path=args.orbitals_file,
                           density_n_samp=10000000000,
                           required_properties=['density'],
                           center_positions=False,
                           radial_coeffs_file=args.radial_coeffs_file,
                           L0_coeffs_file=args.L0_coeffs_file,
                           dtype=args.dtype,
                           grid_fn=grid_fn,
                           sampling_fn=sampling_fn,
                           grid_extent=grid_extent,
                           grid_origin=grid_origin,
                           verbose=args.verbose)

model_init = load_model(args, dataset)

type dtype <class 'torch.dtype'>
args np dir datasets/ethanol_dft_train.npy
best_model_path best_Gtxtn1xI.pth
model code: Gtxtn1xI
args use gpu False
loading density fromdatasets/ethanol_pyscf_def2svp_dft_f.npy...
loading atoms fromdatasets/ethanol_dft_train.npy...
Starting atomsdata density init
Some variables
atoms keys dict_keys(['positions', 'energy', 'forces', 'atom_numbers', 'atom_types'])
level 2
dataset init grid_spec type torch.FloatTensor
finished init
cg_matrix shape torch.Size([121, 121, 121])
self order [1, 3, 5]
self order [1, 3, 5]
self mixing_order [1, 3, 5]
self mixing_order [1, 3, 5]
orbital_spec [[(6, 11, 0), (6, 8, 1), (6, 6, 2), (6, 4, 3), (6, 3, 4), (6, 2, 5)], [(6, 11, 0), (6, 8, 1), (6, 6, 2), (6, 4, 3), (6, 3, 4), (6, 2, 5)], [(8, 11, 0), (8, 8, 1), (8, 6, 2), (8, 4, 3), (8, 3, 4), (8, 2, 5)], [(1, 5, 0), (1, 4, 1), (1, 4, 2), (1, 3, 3), (1, 2, 4)], [(1, 5, 0), (1, 4, 1), (1, 4, 2), (1, 3, 3), (1, 2, 4)], [(1, 5, 0), (1, 4, 1), (1, 4, 2), (1, 3, 3), (1, 2, 4)],

In [19]:
print(args.restart)

None


In [20]:
ids = np.random.choice(np.arange(len(dataset)), replace=False, size=(10,))
model.eval()
with torch.no_grad():
    samples = dataset.get_properties(ids)
    pred = model(samples)
    pred_old = old_model(samples)
    pred_init = model_init(samples)
    # for j in range(len(pred['spherical_coeffs'])):
    #     print('atom', j)
    #     for orb in pred['spherical_coeffs'][j].keys():
    #         if orb[1] == 0:
    #             print('orb', orb)
    #             print('spherical coeffs', pred['spherical_coeffs'][j][orb])
    print('L0 coeffs pred sum', torch.sum(pred['L0_coeffs'], 1))
                
                
    print('density integral true', torch.sum(samples['density'] * samples['coord_weights'], 1))
    print('density integral pred', torch.sum(pred['density'] * pred['coord_weights'], 1))
    print('density integral pred old', torch.sum(pred_old['density'] * pred_old['coord_weights'], 1))
    print('density integral pred init', torch.sum(pred_init['density'] * pred_init['coord_weights'], 1))
    print('density error pred', torch.sum((pred['density'] - samples['density']) * pred['coord_weights'], 1))
    print('density error pred old', torch.sum((pred_old['density'] - samples['density']) * pred_old['coord_weights'], 1))
    print('density error pred init', torch.sum((pred_init['density'] - samples['density']) * pred_init['coord_weights'], 1))

properties positions type torch.FloatTensor
Memory allocated 0.0
Memory cached 0.0
Memory allocated 0.0
Memory cached 0.0
L0_coeffs comb sum before tensor([235.1617, 235.3465, 234.4905, 234.1970, 235.3239, 234.9569, 235.4056,
        234.5108, 235.4075, 234.5445])
integral scale tensor([1.])
L0_coeffs comb sum after tensor([26.0000, 26.0000, 26.0000, 26.0000, 26.0000, 26.0000, 26.0000, 26.0000,
        26.0000, 26.0000])
density nan tensor(0)
density nan tensor(0)
Memory allocated 0.0
Memory cached 0.0
Memory allocated 0.0
Memory cached 0.0
Memory allocated 0.0
Memory cached 0.0
L0_coeffs comb sum before tensor([25.9780, 25.9679, 25.9758, 25.9710, 25.9742, 25.9765, 25.9815, 25.9750,
        25.9782, 25.9845])
integral scale tensor([1.])
L0_coeffs comb sum after tensor([25.9780, 25.9679, 25.9758, 25.9710, 25.9742, 25.9765, 25.9815, 25.9750,
        25.9782, 25.9845])
density nan tensor(0)
density nan tensor(0)
Memory allocated 0.0
Memory cached 0.0
Memory allocated 0.0
Memory cached 0.0

In [ ]:
L0_coeffs = np.load('datasets/augccpvqzjkfit_init_L0.npy', allow_pickle=True).item()
radial_coeffs = np.load('datasets/augccpvqzjkfit_radial_coeffs_df.npy', allow_pickle=True).item()
print('L0 coeffs keys', L0_coeffs.keys())
print('radial_coeffs keys', radial_coeffs.keys())

In [84]:
print('radial_coeffs O', len(radial_coeffs['O']))
print('L0 spherical', L0_coeffs['spherical_coeffs'][0][(6, 0)].shape)
print('atom types', dataset.atoms['atom_types'])

radial_coeffs O 34
L0 spherical torch.Size([1000, 1, 1, 11])
atom types ['C' 'C' 'O' 'H' 'H' 'H' 'H' 'H' 'H']


In [90]:
print('L0 spherical', torch.mean(L0_coeffs['spherical_coeffs'][0][(6, 0)], 0))
print('L0 spherical', torch.mean(L0_coeffs['spherical_coeffs'][1][(6, 0)], 0))
print('L0 spherical', torch.mean(L0_coeffs['spherical_coeffs'][3][(1, 0)], 0))
print('L0 spherical', torch.mean(L0_coeffs['spherical_coeffs'][4][(1, 0)], 0))
print('L0 spherical', torch.mean(L0_coeffs['spherical_coeffs'][5][(1, 0)], 0))
print('L0 spherical', torch.mean(L0_coeffs['spherical_coeffs'][6][(1, 0)], 0))
print('L0 spherical', torch.mean(L0_coeffs['spherical_coeffs'][7][(1, 0)], 0))
print('L0 spherical', torch.mean(L0_coeffs['spherical_coeffs'][8][(1, 0)], 0))
print('L0 spherical shape', L0_coeffs['spherical_coeffs'][0][(6, 0)].shape)

L0 spherical tensor([[[ 1.1699,  0.4176,  0.9394, -0.8756,  0.6810,  1.0179,  1.8026,
           0.4563,  1.4137,  0.0722,  0.0779]]])
L0 spherical tensor([[[ 1.1759,  0.4343,  0.9351, -0.8816,  0.6623,  1.0302,  1.8354,
           0.4593,  1.4413,  0.0808,  0.0919]]])
L0 spherical tensor([[[ 0.5035, -0.9885,  0.2784,  0.2738, -0.0147]]])
L0 spherical tensor([[[ 0.5039, -0.9889,  0.2782,  0.2736, -0.0145]]])
L0 spherical tensor([[[ 0.5000, -0.9784,  0.2847,  0.2577, -0.0136]]])
L0 spherical tensor([[[ 0.4998, -0.9790,  0.2856,  0.2594, -0.0136]]])
L0 spherical tensor([[[ 0.5003, -0.9798,  0.2856,  0.2602, -0.0132]]])
L0 spherical tensor([[[ 0.4858, -0.9064,  0.3745,  0.3048, -0.0016]]])
L0 spherical shape torch.Size([1000, 1, 1, 11])


In [100]:
import ase.data
atom_type_sph = {}
atom_type_width = {}
atom_type_scale = {}
for i in range(len(L0_coeffs['spherical_coeffs'])):
    for key in L0_coeffs['spherical_coeffs'][i].keys():
        atom_type = ase.data.chemical_symbols[key[0]]
        if atom_type not in atom_type_sph.keys():
            atom_type_sph[atom_type] = torch.mean(L0_coeffs['spherical_coeffs'][i][key], 0)
            atom_type_width[atom_type] = torch.mean(L0_coeffs['radial_width'][i][key], 0)
            atom_type_scale[atom_type] = torch.mean(L0_coeffs['radial_scale'][i][key], 0)
            print('atom type sph', atom_type_sph[atom_type].shape)
        else:
            print('atom type sph', atom_type_sph[atom_type].shape)
            atom_type_sph[atom_type] = torch.cat([atom_type_sph[atom_type], torch.mean(L0_coeffs['spherical_coeffs'][i][key], 0)], 0)
            atom_type_width[atom_type] = torch.cat([atom_type_width[atom_type], torch.mean(L0_coeffs['radial_width'][i][key], 0)], 0)
            atom_type_scale[atom_type] = torch.cat([atom_type_scale[atom_type], torch.mean(L0_coeffs['radial_scale'][i][key], 0)], 0)

for t in atom_type_sph.keys():
    atom_type_sph[t] = torch.mean(atom_type_sph[t], 0)
    atom_type_width[t] = torch.mean(atom_type_width[t], 0)
    atom_type_scale[t] = torch.mean(atom_type_scale[t], 0)
    

    
L0_init = {}
L0_init['spherical_coeffs'] = atom_type_sph
L0_init['radial_width'] = atom_type_width
L0_init['radial_scale'] = atom_type_scale


np.save('datasets/augccpvqzjkfit_init_L0.npy', L0_init, allow_pickle=True)

atom type sph torch.Size([1, 1, 11])
atom type sph torch.Size([1, 1, 11])
atom type sph torch.Size([1, 1, 11])
atom type sph torch.Size([1, 1, 5])
atom type sph torch.Size([1, 1, 5])
atom type sph torch.Size([2, 1, 5])
atom type sph torch.Size([3, 1, 5])
atom type sph torch.Size([4, 1, 5])
atom type sph torch.Size([5, 1, 5])


In [102]:
print('init C', L0_init['spherical_coeffs']['C'])

init C tensor([[ 1.1729,  0.4259,  0.9373, -0.8786,  0.6717,  1.0241,  1.8190,  0.4578,
          1.4275,  0.0765,  0.0849]])


In [157]:
print(L0_width.shape)

torch.Size([10, 63])
